In [1]:
import pandas as pd
import numpy as np

Obsługa **indeksowania hierarchicznego** jest ważnym elementem biblioteki pandas umożliwiającym
przypisanie do jednej osi wielu poziomów indeksowania (przypisanie dwóch lub więcej indeksów).

In [2]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])

In [3]:
data

a  1    0.638367
   2    0.603224
   3    0.213408
b  1    0.941164
   3    0.833705
c  1    0.764781
   2    0.876059
d  2    0.862252
   3    0.622402
dtype: float64

Wyświetloną czytelnie serią, której indeksem jest obiekt MultiIndex.

In [4]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

Obiekty o indeksie hierarchicznym obsługują tzw. **indeksowanie częściowe**. Indeksowanie to umożliwia zwięzły wybór podzbioru danych

In [5]:
data["b"]

,0
1,0.941164
3,0.833705


In [6]:
data["b":"c"]

b  1    0.941164
   3    0.833705
c  1    0.764781
   2    0.876059
dtype: float64

In [7]:
data.loc[["b", "d"]]

b  1    0.941164
   3    0.833705
d  2    0.862252
   3    0.622402
dtype: float64

## Przykład na danych

In [8]:
# Dane sprzedażowe w dwóch miastach, w dwóch latach
df = pd.DataFrame({
    "miasto":   ["Łódź", "Łódź", "Kraków", "Kraków"],
    "rok":      [2023,   2024,   2023,     2024],
    "sprzedaz": [120,    145,    200,      230]
})
df

,miasto,rok,sprzedaz
0,Łódź,2023,120
1,Łódź,2024,145
2,Kraków,2023,200
3,Kraków,2024,230


In [9]:
df.index

RangeIndex(start=0, stop=4, step=1)

### Zwykły filtrowanie

Za każdym razem: filtr po mieście, filtr po roku, wybór kolumny. Trzy operacje dla jednej wartości.

In [10]:
# Sprzedaż w Łodzi w 2024 - działa, ale rozwlekle:
df[(df.miasto == "Łódź") & (df.rok == 2024)]["sprzedaz"]

,sprzedaz
1,145


In [11]:
# Sprzedaż w Krakowie w 2023 - znowu to samo:
df[(df.miasto == "Kraków") & (df.rok == 2023)]["sprzedaz"]

,sprzedaz
2,200


### MultiIndex - indeks jako "ścieżka"

In [12]:
# Ten sam zbiór, ale z hierarchicznym indeksem:
sprzedaz = pd.Series(
    [120, 145, 200, 230],
    index=[["Łódź", "Łódź", "Kraków", "Kraków"],
           [2023,   2024,   2023,     2024]]
)
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [13]:
# Sam indeks to obiekt MultiIndex:
sprzedaz.index

MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           )

### Indeksowanie - podstawowe wzorce

In [14]:
# Pojedyncza wartość - krotka (miasto, rok):
sprzedaz["Łódź", 2024]

np.int64(145)

In [15]:
# Wszystko dla Łodzi (wybór zewnętrznego poziomu):
sprzedaz["Łódź"]

,0
2023,120
2024,145


In [16]:
# Wszystkie miasta, ale tylko rok 2023 (wewnętrzny poziom):
sprzedaz.loc[:, 2023]

,0
Łódź,120
Kraków,200


In [17]:
# Zakres miast (uwaga: wymaga posortowanego indeksu):
sprzedaz.sort_index().loc["Kraków":"Łódź"]

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64

## Z Series do DataFrame i z powrotem

In [18]:
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [19]:
# unstack - "wypchnięcie" wewnętrznego poziomu do kolumn:
tabela = sprzedaz.unstack()
print(tabela)
print(type(tabela))

        2023  2024
Kraków   200   230
Łódź     120   145
<class 'pandas.core.frame.DataFrame'>


In [20]:
# stack - operacja odwrotna: kolumny wracają do indeksu:
tab_stack = tabela.stack()
print(tab_stack)
print(type(tab_stack))

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64
<class 'pandas.core.series.Series'>


In [21]:
print(df)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


In [22]:
# Konwersja płaskiej ramki na hierarchiczną - set_index:
df_hier = df.set_index(["miasto", "rok"])
print(df_hier)
print(df_hier.index)

             sprzedaz
miasto rok           
Łódź   2023       120
       2024       145
Kraków 2023       200
       2024       230
MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           names=['miasto', 'rok'])


In [23]:
# I z powrotem - reset_index:
df2 = df_hier.reset_index()
print(df2)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


### Tabele przestawne (ang. _Pivot Table_)

In [26]:
df2_my_pivot = df.set_index(["miasto", "rok"])["sprzedaz"].unstack()
print(df2_my_pivot)

rok     2023  2024
miasto            
Kraków   200   230
Łódź     120   145


In [27]:
df_pivot = pd.pivot_table(df,
                         index="miasto",
                         columns="rok",
                         values="sprzedaz")
print(df_pivot)

rok      2023   2024
miasto              
Kraków  200.0  230.0
Łódź    120.0  145.0


In [28]:
df_kwartaly = pd.DataFrame({
    "miasto":   ["Łódź"]*4 + ["Kraków"]*4,
    "rok":      [2023]*4 + [2023]*4,
    "kwartal":  ["Q1","Q2","Q3","Q4"] * 2,
    "sprzedaz": [30, 25, 35, 30, 50, 45, 55, 50]
})

df_kwartaly_sum = pd.pivot_table(df_kwartaly,
                                index="miasto",
                                columns="rok",
                                values="sprzedaz",
                                aggfunc="sum")
print(df_kwartaly_sum)

rok     2023
miasto      
Kraków   200
Łódź     120


### Agregacje po poziomie

In [29]:
sprzedaz.index.names = ["miasto", "rok"]
sprzedaz.groupby(level="miasto").sum()

,0
miasto,
Kraków,430
Łódź,265


In [30]:
sprzedaz.groupby(level="rok").mean()

,0
rok,
2023,160.0
2024,187.5


### Agregacja w DataFrame z MultiIndex

In [31]:
frame = pd.DataFrame({
    "sprzedaz": [120, 145, 200, 230],
    "koszty":   [80,  90,  130, 140]
}, index=[["Łódź","Łódź","Kraków","Kraków"],
          [2023,  2024,  2023,    2024]])
frame.index.names = ["miasto", "rok"]
frame.groupby(level="miasto").sum()

,sprzedaz,koszty
miasto,,
Kraków,430,270
Łódź,265,170


In [32]:
frame.groupby(level="miasto").agg(["sum", "mean"])

sprzedaz        koszty       
            sum   mean    sum   mean
miasto                              
Kraków      430  215.0    270  135.0
Łódź        265  132.5    170   85.0

In [33]:
frame["sprzedaz"].groupby(level="miasto").agg(
    total="sum", srednia="mean", rozstep=lambda x: x.max()-x.min()
    )

,total,srednia,rozstep
miasto,,,
Kraków,430,215.0,30
Łódź,265,132.5,25


# *Zadania*

Sieć "ModaŁódź" ma sklepy w trzech miastach. Każdy sklep sprzedaje trzy kategorie produktów. Dane obejmują 4 kwartały 2023 i 2024 roku.

## **Zadanie 1 — Budowanie MultiIndex**

a) Utwórz ramkę `df_hi` z hierarchicznym indeksem (`miasto`, `kategoria`, `rok`, `kwartal`). Posortuj indeks.

b) Wyświetl liczbę poziomów indeksu i ich nazwy.

c) Ile unikalnych kombinacji indeksu istnieje? Użyj `.index` do odpowiedzi.

In [34]:
import pandas as pd
import numpy as np

# Wczytanie pliku
df = pd.read_csv("sklepy_moda.csv")

print("ZADANIE 1")
# a) Utworzenie ramki df_hi z hierarchicznym indeksem i posortowanie
df_hi = df.set_index(['miasto', 'kategoria', 'rok', 'kwartal']).sort_index()

# b) Liczba poziomów i ich nazwy
print(f"Liczba poziomów indeksu: {df_hi.index.nlevels}")
print(f"Nazwy poziomów: {df_hi.index.names}")

# c) Unikalne kombinacje indeksu
unikalne_kombinacje = len(df_hi.index.unique())
print(f"Liczba unikalnych kombinacji: {unikalne_kombinacje}")

# Podgląd
display(df_hi.head())

ZADANIE 1
Liczba poziomów indeksu: 4
Nazwy poziomów: ['miasto', 'kategoria', 'rok', 'kwartal']
Liczba unikalnych kombinacji: 72


Unnamed: 0  przychod_tys  koszt_tys  sztuki
miasto kategoria rok  kwartal                                             
Kraków Damska    2023 Q1               24          71.3       51.9     806
                      Q2               25          90.8       52.1    1179
                      Q3               26          77.4       54.5     886
                      Q4               27         116.9       76.5    1285
                 2024 Q1               28          73.0       44.7     843

## **Zadanie 2 — Selekcje na MultiIndex**

a) Wybierz wszystkie dane dla Łodzi.
    
b) Wybierz dane dla Łodzi, kategorii Damska.
    
c) Wybierz dane dla wszystkich miast, ale tylko rok 2024. (Wskazówka: użyj metodę `xs()`)

d) Wybierz Q4 z obu lat, ale tylko dla Krakowa i kategorii Męska.

In [35]:
print("ZADANIE 2")

# a) Wszystkie dane dla Łodzi
print("\na) Dane dla Łodzi:")
display(df_hi.loc['Łódź'].head())

# b) Dane dla Łodzi, kategorii Damska
print("\nb) Dane dla Łodzi, kategoria Damska:")
display(df_hi.loc[('Łódź', 'Damska')].head())

# c) Wszyscy z 2024 roku (używając metody xs)
print("\nc) Wszystkie miasta, rok 2024:")
display(df_hi.xs(2024, level='rok').head())

# d) Q4 z obu lat dla Krakowa i kategorii Męska
print("\nd) Kraków, Męska, tylko Q4 (używamy pd.IndexSlice):")
idx = pd.IndexSlice
display(df_hi.loc[idx['Kraków', 'Męska', :, 'Q4'], :])

ZADANIE 2

a) Dane dla Łodzi:


Unnamed: 0  przychod_tys  koszt_tys  sztuki
kategoria rok  kwartal                                             
Damska    2023 Q1                0          50.5       35.2     615
               Q2                1          59.3       34.5     539
               Q3                2          61.9       34.3     915
               Q4                3          81.8       58.6     775
          2024 Q1                4          49.5       30.2     577


b) Dane dla Łodzi, kategoria Damska:


Unnamed: 0  przychod_tys  koszt_tys  sztuki
rok  kwartal                                             
2023 Q1                0          50.5       35.2     615
     Q2                1          59.3       34.5     539
     Q3                2          61.9       34.3     915
     Q4                3          81.8       58.6     775
2024 Q1                4          49.5       30.2     577


c) Wszystkie miasta, rok 2024:


Unnamed: 0  przychod_tys  koszt_tys  sztuki
miasto kategoria kwartal                                             
Kraków Damska    Q1               28          73.0       44.7     843
                 Q2               29          89.2       65.3     869
                 Q3               30          89.0       53.0     759
                 Q4               31         115.3       70.1    1052
       Dziecięca Q1               44          48.1       28.7     629


d) Kraków, Męska, tylko Q4 (używamy pd.IndexSlice):


Unnamed: 0  przychod_tys  koszt_tys  sztuki
miasto kategoria rok  kwartal                                             
Kraków Męska     2023 Q4               35          89.4       63.8    1253
                 2024 Q4               39         100.3       62.5    1484

## **Zadanie 3 — pivot_table: roczne podsumowanie**

a) Utwórz tabelę przestawną `tab_miasta`, która pokaże **sumę przychodów** w wierszach per miasto, w kolumnach per rok.

b) Dodaj do tabeli kolumnę `zmiana_proc` — procentową zmianę przychodu między 2023 a 2024. Które miasto rosło najszybciej?

c) Utwórz drugą tabelę przestawną, w której wiersze to (miasto, kategoria), kolumny to rok, wartości to *średni przychód kwartalny*. Która kombinacja (miasto, kategoria) ma najwyższy średni przychód w 2024?

In [36]:
print("ZADANIE 3")

# a) Tabela przestawna: wiersze=miasto, kolumny=rok, suma przychodów
tab_miasta = pd.pivot_table(
    df,
    index="miasto",
    columns="rok",
    values="przychod_tys",
    aggfunc="sum")

# b) Procentowa zmiana przychodu
tab_miasta['zmiana_proc'] = ((tab_miasta[2024] - tab_miasta[2023]) / tab_miasta[2023]) * 100
naj_wzrost = tab_miasta['zmiana_proc'].idxmax()

print("\na i b) Tabela sumy przychodów ze zmianą procentową:")
display(tab_miasta)
print(f"-> Najszybciej rosnące miasto to: {naj_wzrost}")

# c) Tabela przestawna: wiersze=(miasto, kategoria), kolumny=rok, średni przychód
tab_miasta_kat = pd.pivot_table(
    df,
    index=["miasto", "kategoria"],
    columns="rok",
    values="przychod_tys",
    aggfunc="mean")
kombinacja_max_2024 = tab_miasta_kat[2024].idxmax()

print("\nc) Tabela średniego przychodu:")
display(tab_miasta_kat)
print(f"-> Kombinacja z najwyższym średnim przychodem w 2024 to: {kombinacja_max_2024}")

ZADANIE 3

a i b) Tabela sumy przychodów ze zmianą procentową:


rok,2023,2024,zmiana_proc
miasto,,,
Kraków,824.3,904.0,9.668810
Wrocław,697.7,753.4,7.983374
Łódź,570.9,637.3,11.630758


-> Najszybciej rosnące miasto to: Łódź

c) Tabela średniego przychodu:


rok                  2023    2024
miasto  kategoria                
Kraków  Damska     89.100  91.625
        Dziecięca  46.700  58.775
        Męska      70.275  75.600
Wrocław Damska     70.775  78.275
        Dziecięca  41.600  46.475
        Męska      62.050  63.600
Łódź    Damska     63.375  62.775
        Dziecięca  32.175  41.900
        Męska      47.175  54.650

-> Kombinacja z najwyższym średnim przychodem w 2024 to: ('Kraków', 'Damska')


## **Zadanie 4 — Agregacja po poziomach**
a) Na `df_hi` oblicz sumę przychodów per miasto (agreguj po poziomie `"miasto"`).

b) Oblicz **średni przychód kwartalny per kategoria** (agreguj po poziomie `"kategoria"`).

c) Użyj `.agg(["sum", "mean", "max"])` na kolumnie `przychod_tys` pogrupowanej po `(miasto, rok)`. Który wiersz ma najwyższą wartość `max`?

In [37]:
print("ZADANIE 4")

# a) Suma przychodów per miasto (agregacja po poziomie miasto)
suma_per_miasto = df_hi.groupby(level="miasto")["przychod_tys"].sum()
print("\na) Suma przychodów per miasto:")
print(suma_per_miasto)

# b) Średni przychód kwartalny per kategoria
srednia_per_kat = df_hi.groupby(level="kategoria")["przychod_tys"].mean()
print("\nb) Średni przychód per kategoria:")
print(srednia_per_kat)

# c) Agregacje wielu statystyk po (miasto, rok)
agg_wielokrotna = df_hi.groupby(level=["miasto", "rok"])["przychod_tys"].agg(["sum", "mean", "max"])
wiersz_max = agg_wielokrotna["max"].idxmax()

print("\nc) Statystyki sum, mean, max:")
display(agg_wielokrotna)
print(f"-> Wiersz (miasto, rok) z absolutnie najwyższym jednym kwartałem (max) to: {wiersz_max}")

ZADANIE 4

a) Suma przychodów per miasto:
miasto
Kraków     1728.3
Wrocław    1451.1
Łódź       1208.2
Name: przychod_tys, dtype: float64

b) Średni przychód per kategoria:
kategoria
Damska       75.987500
Dziecięca    44.604167
Męska        62.225000
Name: przychod_tys, dtype: float64

c) Statystyki sum, mean, max:


sum       mean    max
miasto  rok                          
Kraków  2023  824.3  68.691667  116.9
        2024  904.0  75.333333  115.3
Wrocław 2023  697.7  58.141667   96.0
        2024  753.4  62.783333  107.6
Łódź    2023  570.9  47.575000   81.8
        2024  637.3  53.108333   85.8

-> Wiersz (miasto, rok) z absolutnie najwyższym jednym kwartałem (max) to: ('Kraków', np.int64(2023))


## **Zadanie 5 — Marża i ranking**
a) W `df_hi` dodaj kolumnę `marza_tys = przychod_tys − koszt_tys`.

b) Utwórz tabelę przestawną: *wiersze = miasto*, *kolumny = kategoria*, *wartości = suma marży*. Które miasto jest najbardziej zyskowne? Która kategoria generuje najwyższą marżę?

In [38]:
print("ZADANIE 5")

# a) Dodanie kolumny marży
df_hi['marza_tys'] = df_hi['przychod_tys'] - df_hi['koszt_tys']
print("a) Dodano kolumnę 'marza_tys'. Podgląd pierwszych 3 wierszy:")
display(df_hi.head(3))

# b) Tabela przestawna na marży
tab_marza = pd.pivot_table(
    df_hi.reset_index(),
    index="miasto",
    columns="kategoria",
    values="marza_tys",
    aggfunc="sum")

# Kto zarabia najwięcej? (Suma po wierszach i kolumnach)
zysk_miasta = tab_marza.sum(axis=1) # dodajemy wartości w poprzek kolumn
zysk_kategorie = tab_marza.sum(axis=0) # dodajemy wartości z góry na dół

najzysk_miasto = zysk_miasta.idxmax()
najzysk_kategoria = zysk_kategorie.idxmax()

print("\nb) Tabela sumy marży:")
display(tab_marza)
print(f"-> Najbardziej zyskowne miasto (całkowita marża): {najzysk_miasto}")
print(f"-> Kategoria generująca najwyższą marżę: {najzysk_kategoria}")

ZADANIE 5
a) Dodano kolumnę 'marza_tys'. Podgląd pierwszych 3 wierszy:


Unnamed: 0  przychod_tys  koszt_tys  sztuki  \
miasto kategoria rok  kwartal                                                
Kraków Damska    2023 Q1               24          71.3       51.9     806   
                      Q2               25          90.8       52.1    1179   
                      Q3               26          77.4       54.5     886   

                               marza_tys  
miasto kategoria rok  kwartal             
Kraków Damska    2023 Q1            19.4  
                      Q2            38.7  
                      Q3            22.9


b) Tabela sumy marży:


kategoria,Damska,Dziecięca,Męska
miasto,,,
Kraków,254.8,155.6,201.6
Wrocław,203.0,112.9,183.9
Łódź,184.6,106.8,156.3


-> Najbardziej zyskowne miasto (całkowita marża): Kraków
-> Kategoria generująca najwyższą marżę: Damska
